|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>Sampling<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: a batched sampler you can prove is correct<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

Write the batched sampler. Give each request its own seed. Write a test that
can catch a wrong seed.

This is stage 13. You saw the performance part already. This part is about
correctness, which is harder to check. A broken sampler writes perfect
English.

In [ ]:
### run this cell

device = 'cuda' if torch.cuda.is_available() else 'cpu'
VOCAB = 151936            # Qwen3

def random_batch(batch_size):
  """-> (logits, temperatures, top_ps), one row for each request."""
  return (torch.randn(batch_size, VOCAB, device=device),
          torch.rand(batch_size, device=device)*1.5 + 0.2,
          torch.rand(batch_size, device=device)*0.3 + 0.7)

torch.manual_seed(0)
print(f'vocabulary {VOCAB:,}')

# Exercise 1: one pass, one seed per request

`torch.multinomial` draws from a single global generator, so a batch is not
independently reproducible. Do the inverse-CDF draw yourself.

In [ ]:
def top_p_probs(sorted_logits, top_ps):
  """Softmax over logits sorted from high to low. Keep the top-p nucleus,
  with the token that crosses p. -> the renormalized probabilities."""
  probs = F.softmax(sorted_logits, dim=-1)
  cumulative = probs.cumsum(dim=-1)
  probs = probs * ((cumulative - probs) < top_ps[..., None])
  return probs / probs.sum(dim=-1, keepdim=True)

def sample(logits, temperatures, top_ps, seeds, top_k=64):
  """One vectorized pass, with a seed for each request.

  torch.multinomial draws from ONE global generator, so you cannot reproduce
  one request of a batch alone. Do the inverse-CDF draw yourself, with one
  uniform number for each row from the seed of that row.
  """
  device = logits.device
  sorted_logits, order = torch.topk(logits / temperatures[:, None], top_k, dim=-1)
  probs = top_p_probs(sorted_logits, top_ps)
  # One uniform number for each row, each from its own generator.
  uniforms = 
  # The inverse CDF: the first index where the cumulative mass passes the number.
  picked = 
  return order.gather(1, picked[:, None]).squeeze(1)

logits, temperatures, top_ps = random_batch(8)
seeds = torch.arange(8)
first = sample(logits, temperatures, top_ps, seeds)
second = sample(logits, temperatures, top_ps, seeds)
print('same seeds give the same tokens:', torch.equal(first, second))
print('different seeds differ:         ',
      not torch.equal(first, sample(logits, temperatures, top_ps, seeds + 100)))

# Exercise 2: does it sample the right distribution?

Four tokens, known probabilities, twenty thousand draws. The histogram is the
test.

In [ ]:
# A distribution that you know exactly, sampled many times.
NUM_DRAWS = 20000
true_probs = torch.tensor([0.5, 0.3, 0.15, 0.05], device=device)
logits = true_probs.log()[None, :].repeat(NUM_DRAWS, 1)
temperatures = torch.ones(NUM_DRAWS, device=device)
top_ps = torch.ones(NUM_DRAWS, device=device)    # top_p = 1: no truncation
seeds = torch.arange(NUM_DRAWS)

def empirical_distribution(tokens, num_tokens):
  """-> the fraction of the draws that picked each token."""
  

def print_comparison(expected, empirical):
  print(f"{'token':>6} {'expected':>9} {'sampled':>9}")
  for token, (wanted, got) in enumerate(zip(expected.tolist(), empirical.tolist())):
    print(f'{token:>6} {wanted:>9.3f} {got:>9.3f}')
  print(f'\nmax error {(empirical-expected).abs().max():.4f}')

drawn = 
print_comparison(true_probs, empirical_distribution(drawn, 4))

# Exercise 3: is top-p exact?

Work out on paper which tokens `top_p = 0.9` keeps and what the renormalised
distribution over them is. Then check that is what you drew.

In [ ]:
TOP_P = 0.9
top_ps = torch.full((NUM_DRAWS,), TOP_P, device=device)
drawn = 
# Find by hand which tokens top_p=0.9 must keep, and the renormalized
# distribution over them.
exact = 
print_comparison(exact, empirical_distribution(drawn, 4))

### Before you open the solution

1. Change `(cumulative - probs) < top_ps` to `cumulative < top_ps` and run Exercise 3 again.
   Which tokens survive now? Which result is correct?
2. Your Exercise 2 histogram must agree to approximately 1%. What does a
   missing renormalisation after the top-p mask do to it? Do you see the
   difference in the generated text?
3. One generator for each row is slow. For a batch of 256 that is 256 Python
   objects for each step. What do you keep between steps instead?